# IndoMultiDomain-Core V1 — Final Audit

Audit ini **tidak mengubah corpus**. Ia memeriksa schema, integritas ID, distribusi domain/source/genre,
panjang teks, sinyal bahasa/script, kemungkinan PII, overlap trivial setelah normalisasi, provenance,
license completeness, dan split integrity.

Semua output ditulis ke `04_analysis/final_audit/`.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
audit_code='\nfrom pathlib import Path\nimport pandas as pd\nimport numpy as np\nimport re, json, hashlib, unicodedata\nfrom collections import Counter\n\nROOT=Path(\'/content/drive/MyDrive/IndoMultiDomain\')\nCORE=ROOT/\'03_harmonized\'/\'indomultidomain_core_v1.parquet\'\nOUT=ROOT/\'04_analysis\'/\'final_audit\'\nOUT.mkdir(parents=True,exist_ok=True)\n\ndf=pd.read_parquet(CORE)\n\nrequired=[\n"imd_id","text","domain","subdomain","genre","source_id","source_repository",\n"source_dataset","source_version","source_platform","original_id","language","year",\n"original_task","original_label","original_label_type","license","provenance_note",\n"char_count","word_count","quality_flag","duplicate_group","split"\n]\n\n# 1. Schema & null audit\nschema_rows=[]\nfor c in required:\n    schema_rows.append({\n        "column":c,\n        "present":c in df.columns,\n        "dtype":str(df[c].dtype) if c in df.columns else None,\n        "null_count":int(df[c].isna().sum()) if c in df.columns else None,\n        "null_pct":round(float(df[c].isna().mean()*100),4) if c in df.columns else None,\n    })\npd.DataFrame(schema_rows).to_csv(OUT/\'schema_audit.csv\',index=False)\n\n# 2. Core integrity\nintegrity={\n    "records":int(len(df)),\n    "unique_imd_id":int(df.imd_id.nunique()),\n    "duplicate_imd_id":int(df.imd_id.duplicated().sum()),\n    "unique_text_hash":int(df.duplicate_group.nunique()),\n    "duplicate_text_hash":int(df.duplicate_group.duplicated().sum()),\n    "empty_text":int(df.text.astype(str).str.strip().eq("").sum()),\n    "domains":int(df.domain.nunique()),\n    "subdomains":int(df.subdomain.nunique()),\n    "genres":int(df.genre.nunique()),\n    "sources":int(df.source_id.nunique()),\n    "total_words":int(df.word_count.sum()),\n    "total_chars":int(df.char_count.sum()),\n}\n(OUT/\'integrity_summary.json\').write_text(json.dumps(integrity,indent=2),encoding=\'utf-8\')\n\n# 3. Distribution tables\ndf.groupby([\'source_id\',\'source_dataset\',\'domain\',\'genre\'],dropna=False).agg(\n    records=(\'imd_id\',\'count\'),\n    words=(\'word_count\',\'sum\'),\n    median_words=(\'word_count\',\'median\'),\n    mean_words=(\'word_count\',\'mean\'),\n    min_words=(\'word_count\',\'min\'),\n    max_words=(\'word_count\',\'max\')\n).reset_index().to_csv(OUT/\'source_distribution.csv\',index=False)\n\ndf.groupby([\'domain\'],dropna=False).agg(\n    records=(\'imd_id\',\'count\'),\n    words=(\'word_count\',\'sum\'),\n    median_words=(\'word_count\',\'median\'),\n    sources=(\'source_id\',\'nunique\'),\n    genres=(\'genre\',\'nunique\')\n).reset_index().sort_values(\'records\',ascending=False).to_csv(OUT/\'domain_distribution.csv\',index=False)\n\ndf.groupby([\'genre\'],dropna=False).agg(\n    records=(\'imd_id\',\'count\'),\n    words=(\'word_count\',\'sum\'),\n    median_words=(\'word_count\',\'median\'),\n    sources=(\'source_id\',\'nunique\'),\n    domains=(\'domain\',\'nunique\')\n).reset_index().sort_values(\'records\',ascending=False).to_csv(OUT/\'genre_distribution.csv\',index=False)\n\ndf.groupby([\'split\'],dropna=False).agg(\n    records=(\'imd_id\',\'count\'),\n    sources=(\'source_id\',\'nunique\'),\n    domains=(\'domain\',\'nunique\')\n).reset_index().to_csv(OUT/\'split_distribution.csv\',index=False)\n\ndf.groupby([\'quality_flag\'],dropna=False).size().reset_index(name=\'records\').to_csv(OUT/\'quality_flags.csv\',index=False)\n\n# 4. Length audit\nbins=[0,5,10,20,50,100,200,500,1000,10**9]\nlabels=[\'1-5\',\'6-10\',\'11-20\',\'21-50\',\'51-100\',\'101-200\',\'201-500\',\'501-1000\',\'>1000\']\ndf[\'_word_bin\']=pd.cut(df.word_count,bins=bins,labels=labels,include_lowest=True,right=True)\ndf.groupby([\'_word_bin\'],observed=False).size().reset_index(name=\'records\').to_csv(OUT/\'length_distribution.csv\',index=False)\n\n# suspicious very short/very long examples\ndf.nsmallest(100,\'word_count\')[[\'imd_id\',\'source_id\',\'domain\',\'genre\',\'word_count\',\'text\']].to_csv(OUT/\'shortest_100.csv\',index=False)\ndf.nlargest(100,\'word_count\')[[\'imd_id\',\'source_id\',\'domain\',\'genre\',\'word_count\',\'text\']].to_csv(OUT/\'longest_100.csv\',index=False)\n\n# 5. Character/script heuristics — QC signal only, not semantic annotation\ndef script_stats(t):\n    t=str(t)\n    letters=[ch for ch in t if ch.isalpha()]\n    if not letters:\n        return (0.0,0.0)\n    ascii_lat=sum((\'a\'<=ch.lower()<=\'z\') for ch in letters)\n    return ascii_lat/len(letters), len(letters)\n\nvals=df.text.map(script_stats)\ndf[\'_latin_ratio\']=[x[0] for x in vals]\ndf[\'_letter_count\']=[x[1] for x in vals]\nflags=df[(df._letter_count>=10) & (df._latin_ratio<0.75)][\n    [\'imd_id\',\'source_id\',\'domain\',\'genre\',\'_latin_ratio\',\'text\']\n]\nflags.to_csv(OUT/\'script_language_review_candidates.csv\',index=False)\n\n# 6. URL/email/phone/direct identifier signals\npatterns={\n    \'contains_email\':r\'[\\w.+-]+@[\\w.-]+\\.[A-Za-z]{2,}\',\n    \'contains_url\':r\'https?://|www\\.\',\n    \'contains_phone_candidate\':r\'(?<!\\d)(?:\\+?62|0)8\\d{7,12}(?!\\d)\',\n}\npii_summary=[]\nfor name,pat in patterns.items():\n    mask=df.text.astype(str).str.contains(pat,regex=True,case=False,na=False)\n    pii_summary.append({\'signal\':name,\'records\':int(mask.sum())})\n    if mask.any():\n        df.loc[mask,[\'imd_id\',\'source_id\',\'domain\',\'genre\',\'text\']].head(500).to_csv(OUT/f\'{name}_examples.csv\',index=False)\npd.DataFrame(pii_summary).to_csv(OUT/\'pii_signal_summary.csv\',index=False)\n\n# 7. Cross-source exact overlap audit (should normally be empty after global dedup,\n# but test normalized lower/punctuation-insensitive key to detect trivial variants)\ndef soft_norm(t):\n    t=unicodedata.normalize(\'NFC\',str(t)).lower()\n    t=re.sub(r\'https?://\\S+\',\' \',t)\n    t=re.sub(r\'[^\\w\\s]\',\' \',t,flags=re.UNICODE)\n    t=re.sub(r\'\\s+\',\' \',t).strip()\n    return t\n\ndf[\'_soft_norm\']=df.text.map(soft_norm)\nsoftdup=df[(df._soft_norm.str.len()>=20) & df.duplicated(\'_soft_norm\',keep=False)].copy()\nsoftdup.sort_values(\'_soft_norm\')[[\'imd_id\',\'source_id\',\'domain\',\'genre\',\'text\',\'_soft_norm\']].to_csv(\n    OUT/\'soft_normalized_duplicate_candidates.csv\',index=False\n)\n\n# 8. Source balance and dominance\nsource_counts=df.source_id.value_counts()\ndomain_counts=df.domain.value_counts()\nbalance={\n    "largest_source":str(source_counts.index[0]),\n    "largest_source_records":int(source_counts.iloc[0]),\n    "largest_source_share_pct":round(float(source_counts.iloc[0]/len(df)*100),2),\n    "largest_domain":str(domain_counts.index[0]),\n    "largest_domain_records":int(domain_counts.iloc[0]),\n    "largest_domain_share_pct":round(float(domain_counts.iloc[0]/len(df)*100),2),\n}\n(OUT/\'balance_summary.json\').write_text(json.dumps(balance,indent=2),encoding=\'utf-8\')\n\n# 9. Provenance/license completeness\nprov=pd.DataFrame({\n    \'field\':[\'license\',\'provenance_note\',\'source_dataset\',\'source_repository\'],\n    \'missing_records\':[\n        int(df.license.fillna(\'\').astype(str).str.strip().eq(\'\').sum()),\n        int(df.provenance_note.fillna(\'\').astype(str).str.strip().eq(\'\').sum()),\n        int(df.source_dataset.fillna(\'\').astype(str).str.strip().eq(\'\').sum()),\n        int(df.source_repository.fillna(\'\').astype(str).str.strip().eq(\'\').sum()),\n    ]\n})\nprov.to_csv(OUT/\'provenance_completeness.csv\',index=False)\n\n# 10. Final PASS/REVIEW gates\nchecks=[\n    (\'imd_id_unique\', df.imd_id.is_unique),\n    (\'exact_text_hash_unique\', df.duplicate_group.is_unique),\n    (\'no_empty_text\', df.text.astype(str).str.strip().ne(\'\').all()),\n    (\'all_required_columns_present\', all(c in df.columns for c in required)),\n    (\'license_complete\', df.license.fillna(\'\').astype(str).str.strip().ne(\'\').all()),\n    (\'provenance_complete\', df.provenance_note.fillna(\'\').astype(str).str.strip().ne(\'\').all()),\n    (\'valid_split_values\', set(df.split.dropna().unique()).issubset({\'train\',\'validation\',\'test\'})),\n]\ncheck_df=pd.DataFrame(checks,columns=[\'check\',\'pass\'])\ncheck_df.to_csv(OUT/\'audit_checks.csv\',index=False)\n\nsummary={\n    **integrity,\n    **balance,\n    "script_language_review_candidates":int(len(flags)),\n    "soft_normalized_duplicate_candidates":int(len(softdup)),\n    "pii_signals":{r[\'signal\']:r[\'records\'] for r in pii_summary},\n    "all_core_checks_pass":bool(check_df[\'pass\'].all())\n}\n(OUT/\'FINAL_AUDIT_SUMMARY.json\').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n\nprint(json.dumps(summary,indent=2,ensure_ascii=False))\nprint("\\\\nAUDIT CHECKS")\nprint(check_df.to_string(index=False))\nprint("\\\\nFiles:",OUT)\n'
from pathlib import Path
p=Path('/content/drive/MyDrive/IndoMultiDomain/06_builder/indomultidomain_final_audit.py')
p.write_text(audit_code,encoding='utf-8')
exec(compile(audit_code,str(p),'exec'))


{
  "records": 69075,
  "unique_imd_id": 69075,
  "duplicate_imd_id": 0,
  "unique_text_hash": 69075,
  "duplicate_text_hash": 0,
  "empty_text": 0,
  "domains": 8,
  "subdomains": 12,
  "genres": 3,
  "sources": 9,
  "total_words": 1418001,
  "total_chars": 9718645,
  "largest_source": "SRC008",
  "largest_source_records": 14999,
  "largest_source_share_pct": 21.71,
  "largest_domain": "health_healthcare",
  "largest_domain_records": 20008,
  "largest_domain_share_pct": 28.97,
  "script_language_review_candidates": 35,
  "soft_normalized_duplicate_candidates": 720,
  "pii_signals": {
    "contains_email": 1,
    "contains_url": 7,
    "contains_phone_candidate": 5
  },
  "all_core_checks_pass": true
}
\nAUDIT CHECKS
                       check  pass
               imd_id_unique  True
      exact_text_hash_unique  True
               no_empty_text  True
all_required_columns_present  True
            license_complete  True
         provenance_complete  True
          valid_split_values